In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

data_dir = '../data/'

In [14]:
df = pd.read_csv(os.path.join(data_dir, 'differential_expression_results.csv'))
df.dtypes

gene        object
log2FC     float64
p_value    float64
dtype: object

In [15]:

# Add a small value to avoid division by zero
df['p_value'] = df['p_value'].replace(0, np.min(df['p_value'][df['p_value'] > 0]))

df['neg_log10_p_value'] = -np.log10(df['p_value'])

log2fc_threshold = 1.5
p_value_threshold = 0.05
neg_log10_p_value_threshold = -np.log10(p_value_threshold)

def categorize_gene(row):
    if row['log2FC'] > log2fc_threshold and row['p_value'] < p_value_threshold:
        return 'Upregulated'
    elif row['log2FC'] < -log2fc_threshold and row['p_value'] < p_value_threshold:
        return 'Downregulated'
    else:
        return 'Not Significant'

df['Gene_Status'] = df.apply(categorize_gene, axis=1)

fig = px.scatter(df,
                 x='log2FC',
                 y='neg_log10_p_value',
                 color='Gene_Status',
                 hover_data=['gene', 'log2FC', 'p_value'],
                 color_discrete_map={
                     'Upregulated': 'red',
                     'Downregulated': 'blue',
                     'Not Significant': 'gray'
                 },
                 title='Interactive Volcano Plot of Differential Gene Expression',
                 labels={
                     'log2FC': 'Log2 Fold Change',
                     'neg_log10_p_value': '-log10(p-value)',
                     'Gene_Status': 'Gene Status'
                 })

fig.add_hline(y=neg_log10_p_value_threshold, line_dash="dash", line_color="black")
fig.add_vline(x=log2fc_threshold, line_dash="dash", line_color="black")
fig.add_vline(x=-log2fc_threshold, line_dash="dash", line_color="black")

fig.show()
fig.write_image('../data/volcano_plot.png', width=2000, height=800)